# P4-5 · RAG nâng cao

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q sentence-transformers

## Lý thuyết

In [ ]:
def rrf(lists, k=60):
    score = {}
    for lst in lists:
        for rank, doc in enumerate(lst, start=1):
            score[doc] = score.get(doc, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

# ví dụ nhỏ: hai bảng xếp hạng của cùng một câu hỏi
bm25_top50 = ["đổi trả 7 ngày", "phí ship", "bảo hành 12 tháng", "đổi size"]
vector_top50 = ["đổi size", "đổi trả 7 ngày", "hoàn tiền", "phí ship"]
candidates = rrf([bm25_top50, vector_top50])[:50]
candidates          # tài liệu đứng cao ở cả hai danh sách được đẩy lên đầu

['đổi trả 7 ngày', 'đổi size', 'phí ship', 'bảo hành 12 tháng', 'hoàn tiền']


In [ ]:
# Cần mạng: tải reranker từ Hugging Face, chạy trên Colab
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")
pairs = [(question, chunks[i]["text"]) for i in candidates]         # 50 cặp
scores = reranker.predict(pairs, batch_size=16)
top5 = [candidates[i] for i in scores.argsort()[::-1][:5]]

## Bài tập 5: Pipeline hai tầng {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
# Cần mạng và index ở bài P4.2: chạy trên Colab
import numpy as np
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

def rrf(lists, k=60):
    score = {}
    for lst in lists:
        for rank, d in enumerate(lst, 1):
            score[d] = score.get(d, 0) + 1 / (k + rank)
    return sorted(score, key=score.get, reverse=True)

def retrieve(question, k_final=5, k_wide=50):
    qv = emb.encode([question], normalize_embeddings=True).astype("float32")
    vec = list(index.search(qv, k_wide)[1][0])
    kw = list(np.argsort(-bm25.get_scores(tokenize(question)))[:k_wide])
    cand = rrf([vec, kw])[:k_wide]
    scores = reranker.predict([(question, chunks[i]["text"]) for i in cand])
    return [cand[i] for i in np.argsort(-scores)[:k_final]]